# Financial Backtesting — NVDA Stock Price Predictions

Backtests frontier LLMs on historical NVIDIA (NVDA) stock price predictions using `ContinuousAnswerType`.

The pipeline:
1. **Download historical prices** from Yahoo Finance via `yfinance`
2. **Create samples** with ground-truth closing prices as labels
3. **Send to 3 models** via QuestionPipeline (GPT-4.1-mini, Claude Sonnet 4, Gemini 2.5 Flash)
4. **Score** — compare each model's numeric price estimates against actual prices
5. **Analyze** — per-model metrics and consensus

In [1]:
%pip install lightningrod-ai python-dotenv pandas yfinance

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Fetch NVDA price data

Download historical daily closing prices for NVIDIA (NVDA) from Yahoo Finance via `yfinance`. No API key needed — Yahoo Finance data is free.

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf

df_prices = yf.download("NVDA", start="2024-07-01", end="2025-01-01")

# Sample ~20 trading days evenly spaced across the range
NUM_SAMPLES = 20
indices = np.linspace(0, len(df_prices) - 1, NUM_SAMPLES, dtype=int)
df_sampled = df_prices.iloc[indices].copy()

# Build a list of dicts for easy downstream use
price_data = []
for date, row in df_sampled.iterrows():
    close = float(row["Close"].iloc[0]) if hasattr(row["Close"], "iloc") else float(row["Close"])
    volume = int(row["Volume"].iloc[0]) if hasattr(row["Volume"], "iloc") else int(row["Volume"])
    price_data.append({
        "date": date,
        "date_str": date.strftime("%B %d, %Y"),
        "close": round(close, 2),
        "volume": volume,
    })

print(f"Sampled {len(price_data)} trading days from {price_data[0]['date_str']} to {price_data[-1]['date_str']}")
print()
print(f"{'Date':<22} {'Close ($)':>10} {'Volume':>15}")
print("-" * 50)
for p in price_data:
    print(f"{p['date_str']:<22} {p['close']:>10.2f} {p['volume']:>15,}")

[*********************100%***********************]  1 of 1 completed

Sampled 20 trading days from July 01, 2024 to December 31, 2024

Date                    Close ($)          Volume
--------------------------------------------------
July 01, 2024              124.25     284,885,500
July 10, 2024              134.85     248,978,600
July 19, 2024              117.88     217,223,800
July 30, 2024              103.69     486,833,300
August 07, 2024             98.87     411,440,400
August 16, 2024            124.53     302,589,900
August 27, 2024            128.25     303,134,600
September 05, 2024         107.16     306,850,700
September 16, 2024         116.74     248,772,300
September 25, 2024         123.47     284,692,900
October 03, 2024           122.81     277,118,000
October 14, 2024           138.02     232,347,700
October 23, 2024           139.51     285,930,000
October 31, 2024           132.71     270,039,600
November 11, 2024          145.21     182,325,600
November 20, 2024          145.84     309,871,700
November 29, 2024          138.20 

## Create samples

Each trading day becomes a `Sample` with:
- `seed_text` = a question asking for NVDA's closing price on that date
- `label` = the actual closing price as a numeric string

In [4]:
from lightningrod import create_sample

samples = []
for p in price_data:
    seed_text = f"What was NVIDIA's (NVDA) closing stock price in USD on {p['date_str']}?"
    
    meta = {
        "date": p["date_str"],
        "volume": p["volume"],
    }
    sample = create_sample(seed_text, str(p["close"]), p["date"], meta)

    samples.append(sample)

print(f"{len(samples)} samples ready for evaluation")

20 samples ready for evaluation


## Upload input dataset

In [5]:
input_dataset = lr.datasets.create_from_samples(samples)
print(f"Created input dataset: {input_dataset.id}")
print(f"Total samples: {input_dataset.num_rows}")

Created input dataset: 9e0c463a-4125-4f5b-ac36-82cb9ef6adff
Total samples: 20


## Configure the pipeline

The pipeline has four stages:
1. **TemplateQuestionGenerator** — fills a financial analyst prompt template with each date's question
2. **QuestionRenderer** — renders the question with a continuous (numeric) answer type
3. **RolloutGenerator** — sends the rendered prompt to multiple LLMs via OpenRouter
4. **RolloutScorer** — scores each model's price estimate against the ground-truth closing price

In [6]:
from lightningrod import (
    QuestionPipeline,
    TemplateQuestionGenerator,
    QuestionRenderer,
    RolloutGenerator,
    RolloutScorer,
    ContinuousAnswerType,
    ModelConfig,
    ModelSourceType,
)

QUESTION_TEMPLATE = (
    "You are a financial analyst with deep knowledge of stock market history. "
    "Your task is to estimate the closing stock price for the given date. "
    "Think step by step about what you know regarding the company's performance, "
    "market conditions, and any major events around that time period. "
    "Then provide your best numeric estimate of the closing price in USD.\n\n"
    "{seed_text}"
)

models = [
    ModelConfig(model_name="openai/gpt-4.1-mini", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
    ModelConfig(model_name="anthropic/claude-sonnet-4", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
    ModelConfig(model_name="google/gemini-2.5-flash", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
]

answer_type = ContinuousAnswerType()

pipeline = QuestionPipeline(
    question_generator=TemplateQuestionGenerator(question_template=QUESTION_TEMPLATE),
    renderer=QuestionRenderer(answer_type=answer_type),
    rollout_generator=RolloutGenerator(models=models),
    scorer=RolloutScorer(answer_type=answer_type),
)

## Run the pipeline

This sends each price question to all three models for numeric estimation. It may take a few minutes depending on the number of samples.

In [7]:
dataset = lr.transforms.run(
    pipeline,
    input_dataset=input_dataset,
    name="NVDA Financial Backtesting",
)

Output()

## Per-model metrics

`mean_reward` for continuous answer types is the continuous log score — a scale-invariant metric where higher values indicate better calibrated numeric predictions.

In [8]:
import pandas as pd
from lightningrod.utils import compute_metrics_summary

result_samples = dataset.download()

summary = compute_metrics_summary(result_samples)
df = pd.DataFrame.from_dict(summary, orient="index")
df.index.name = "model"
df[["mean_reward", "parse_rate", "n_total"]]

,mean_reward,parse_rate,n_total
model,,,
openai/gpt-4.1-mini,-11.764643,1.0,20
anthropic/claude-sonnet-4,-2.342817,1.0,20
google/gemini-2.5-flash,-8.579244,0.4,20


## Consensus analysis

Where do the models agree, and where do they diverge? `compute_consensus` extracts each model's predicted price and computes:
- **spread** — max prediction minus min prediction across models (higher = more disagreement)
- **all_agree** — whether all models' predictions are on the same side of the median prediction

In [9]:
from lightningrod.utils import compute_consensus

consensus = compute_consensus(result_samples)
n_agree = sum(1 for c in consensus if c["all_agree"])
n_total = len(consensus)

print(f"Consensus: {n_agree}/{n_total} questions have full agreement ({n_agree / n_total * 100:.0f}%)")
print(f"Mean spread: ${sum(c['spread'] for c in consensus) / n_total:.2f}")
print()

rows = []
for c in consensus:
    row = {
        "Question": c["question_text"][:80],
        "Actual": c["label"],
        "Spread": round(c["spread"], 2),
        "Agree": c["all_agree"],
    }
    for model, price in c["predictions"].items():
        short_name = model.split("/")[-1] if "/" in model else model
        row[short_name] = round(price, 2)
    rows.append(row)

df_consensus = pd.DataFrame(rows)
df_consensus

Consensus: 20/20 questions have full agreement (100%)
Mean spread: $438.83



,Question,Actual,Spread,Agree,gpt-4.1-mini,claude-sonnet-4,gemini-2.5-flash
0,You are a financial analyst with deep knowledg...,134.85,931.50,True,530.0,118.50,1050.0
1,You are a financial analyst with deep knowledg...,138.2,883.75,True,485.0,141.25,1025.0
2,You are a financial analyst with deep knowledg...,123.47,858.50,True,520.0,121.50,980.0
3,You are a financial analyst with deep knowledg...,139.51,481.50,True,620.0,138.50,NaN
4,You are a financial analyst with deep knowledg...,117.88,427.50,True,555.0,127.50,NaN
5,You are a financial analyst with deep knowledg...,128.25,424.39,True,510.0,125.61,550.0
6,You are a financial analyst with deep knowledg...,135.03,411.50,True,550.0,138.50,NaN
7,You are a financial analyst with deep knowledg...,103.69,387.50,True,505.0,117.50,130.0
8,You are a financial analyst with deep knowledg...,98.87,384.75,True,495.0,110.25,148.5
9,You are a financial analyst with deep knowledg...,145.84,374.50,True,520.0,145.50,NaN


## Next steps

- **Prediction market backtesting**: See [polymarket_backtesting.ipynb](polymarket_backtesting.ipynb) for binary probability forecasting
- **Document classification benchmark**: See [document_classification.ipynb](document_classification.ipynb) for evaluating LLMs on multi-class classification
- **News forecasting consensus**: See [model_consensus.ipynb](model_consensus.ipynb) for generating forecasting questions from news
- **Full API reference**: See [API.md](../../API.md) for all pipeline options and configurations